# Backpropagation

Take a look at the mathematics of the backpropagation algorithm.

We will cover the following:

- Notations

- Forward pass

- Backward pass

- The chain rule for the backward pass

Neural Networks (NN) are non-linear classifiers that can be formulated as a series of matrix multiplications. Just like linear classifiers, they can be trained using the same principles we followed before, namely the gradient descent algorithm. The difficulty arises in computing the gradients. 

But first things first. 

Let’s start with a straightforward example of a two-layered NN, with each layer containing just one neuron.

# Notations

- The superscript defines the layer that we are in.

- $o^{L}$ denotes the activation of layer L.

- $w^{L}$ is a scalar weight of the layer L.

- $b^{L}$ is the bias term of layer L.

- $C$ is the cost function, $t$ is our target class, and $f$ is the activation function.


# Forward pass

Our lovely model would look something like this in a simple sketch:


![pic](https://raw.githubusercontent.com/CUTe-EmbeddedAI/images/main/images/fig09.PNG)

We can write the output of a neuron at layer $L$ as:

> $o^{L} = f(w^{L}o^{L-1} + b^{L})$

To simplify things, let’s define:

> $z^{L} = w^{L}o^{L-1} + b^{L}$

so that our basic equation will become:

> $o^L = f(z^l)$

We also know that our loss function is:

> $C = (o^L − t)^2 $

This is the so-called **forward pass**. 

We take some input and pass it through the network. From the output of the network, we can compute the loss $C$.

# Backward pass

> Backward pass is the process of adjusting the weights $w$ in all the layers to minimize the loss $C$.

To adjust the weights based on the training example, we can use our known **update rule**:

> $w_t^L = w_{t-1}^L - \lambda \frac{\delta C}{\delta w^L} $

where $\lambda$ is the learning rate that scales down the gradient.

It should be clear by now that the only thing left to compute is the gradient $\frac{\delta C}{\delta w^L}$ (the derivative of the loss with respect to the weight).

One way to think about computing $\frac{\delta C}{\delta w^L}$ is through the following diagram, which is called computational graph:


![pic](https://raw.githubusercontent.com/CUTe-EmbeddedAI/images/main/images/fig10.PNG)

We summarize the performed operation in this way. To convert this into math, we need to revisit the chain rule.

# The chain rule for the backward pass

To compute the gradient $\frac{\delta C}{\delta w^L}$, our most useful tool is calculus and the chain rule.

Using both, we can write:
\frac{\delta o^L}{\delta z^L \frac{\delta C}{\delta o^L}}
> $\frac{\delta C}{\delta w^L} = \frac{\delta z^L}{\delta w^L} \frac{\delta o^L}{\delta z^L} \frac{\delta C}{\delta o^L}$

It is evident that the final gradient is affected by the gradients of the previous neuron, which in turn is affected by the gradients of the one before. You can see that in order to compute the gradient, we need to go back (through the chain rule) all the way to the beginning of the network.

In other terms, we need to propagate the error backwards. This is how the backpropagation algorithm got its name.

To find the gradients, let’s compute all the subgradients. By using basic calculus,we get:

>> $C = (o^L - t)^2 \rightarrow \frac{\delta C}{\delta o^L} = 2(o^L - t)$

>> $o^L = f(w^Lo^{L-1} + b^L) = f(z^L) \rightarrow \frac{\delta o^L}{\delta z^L} = f'(o^L)$

>> $z^L = w^Lo^{L-1} + b^L \rightarrow \frac{\delta z^L}{\delta w^L} = o^{L-1}$

Combining them all together, we acquire our final gradient:

>> $ \frac{\delta C}{\delta w^L} = o^{L-1} \ast f'(o^L) \ast 2(o^L - t)$

Similar equations can be derived for the biases. Instead of $\frac{\delta z^L}{\delta b^L}$, we would have:

>> $ z^L = w^Lo^{L-1} + b^L \rightarrow \frac{\delta z^L}{\delta b^L} =1 $ 

For completion, if we do the math, we get:

>> $\frac{\delta C}{\delta b^L} = f'(o^L) \ast 2(o^L - t)$


Now, we can adjust the weight and biases based on a single training example based on the update rule:

>> $w^L_t = w^L_{t-1} - \lambda \frac{\delta C}{\delta w^L}$

Next, we’ll feed the next example and readjust, and repeat. This is the infamous **BACKPROPAGATION**.

You might argue that this is oversimplified because we only have 1 neuron. To be honest, not much will change if we add more neurons per layer. We will essentially conclude to the same equation.


![pic](https://raw.githubusercontent.com/CUTe-EmbeddedAI/images/main/images/fig11.PNG)

Two final things to note here:

- The derivative with respect to the activation is a summation due to the fact that the activation of a neuron now depends on the activations of all th eneurons on the previous layer.

- The same derivative also depends on the derivatives of the next layer’s activation (backpropagation of the error).

You now have a sense of how NNs learn, and that is no easy task.

> **Important note:** We will not be computing gradients in every network that we define. The gradients are computed automatically in modern frameworks such as PyTorch.

No more partial derivatives!

---
## torch.autograd — Let PyTorch Compute Gradients

In practice you never hand-compute partial derivatives. PyTorch's `autograd` engine
does it for you via reverse-mode automatic differentiation — exactly the algorithm
described above, implemented efficiently on a dynamic computation graph.

The key idea: every tensor created with `requires_grad=True` tracks operations
applied to it. Calling `.backward()` walks that graph in reverse and accumulates
gradients in each tensor's `.grad` attribute.

In [ ]:
# [UPDATED] torch.autograd demo — mirrors the math in the theory cells above
import torch

# ── Scalar example ──────────────────────────────────────────────
# Suppose C = (o - t)^2, o = sigmoid(w*x + b)
torch.manual_seed(0)

w = torch.tensor(0.5,  requires_grad=True)
b = torch.tensor(-0.1, requires_grad=True)
x = torch.tensor(1.0)   # input
t = torch.tensor(1.0)   # target

# Forward pass
z = w * x + b
o = torch.sigmoid(z)       # activation
C = (o - t) ** 2           # MSE loss

# Backward pass  (PyTorch fills .grad for every leaf with requires_grad=True)
C.backward()

print(f'w = {w.item():.4f}   dC/dw = {w.grad.item():.6f}')
print(f'b = {b.item():.4f}   dC/db = {b.grad.item():.6f}')

# ── Mini neural network example ─────────────────────────────────
import torch.nn as nn

model = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(8, 1)
)
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

X = torch.randn(16, 4)
y = torch.randn(16, 1)

# One training step
optimizer.zero_grad()        # clear stale gradients
output = model(X)            # forward
loss   = criterion(output, y)
loss.backward()              # backward — gradients populated
optimizer.step()             # weight update

print(f'\nLoss after 1 step: {loss.item():.4f}')

# Inspect gradient norms per layer
for name, param in model.named_parameters():
    if param.grad is not None:
        print(f'  {name:20s} grad norm = {param.grad.norm().item():.6f}')

### Gradient Flow Visualization

A common diagnostic: plot the **gradient norm** at each layer after a backward pass.
Healthy training shows gradients of similar magnitude across all layers.
Vanishing gradients appear as near-zero norms in early layers.

In [ ]:
# [UPDATED] Gradient flow plot — useful debugging tool for deep networks
import matplotlib.pyplot as plt
import torch, torch.nn as nn

torch.manual_seed(42)

# Build a deeper network to show gradient decay
layers = []
dims = [16, 64, 64, 64, 64, 32, 1]
for i in range(len(dims)-1):
    layers += [nn.Linear(dims[i], dims[i+1]), nn.Sigmoid()]  # sigmoid causes vanishing
deep_net = nn.Sequential(*layers)

X = torch.randn(32, 16)
y = torch.randn(32, 1)
loss = nn.MSELoss()(deep_net(X), y)
loss.backward()

# Collect gradient norms for weight matrices only
grad_norms, names = [], []
for name, p in deep_net.named_parameters():
    if 'weight' in name and p.grad is not None:
        grad_norms.append(p.grad.norm().item())
        names.append(name.replace('.weight',''))

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(names, grad_norms, color=['#e74c3c' if g < 1e-4 else '#2ecc71' for g in grad_norms])
ax.set_xlabel('Layer'); ax.set_ylabel('Gradient Norm (log scale)')
ax.set_yscale('log'); ax.set_title('Gradient Flow — Vanishing Gradient with Sigmoid')
ax.axhline(1e-4, color='k', linestyle='--', label='Vanishing threshold')
ax.legend(); plt.tight_layout(); plt.show()
print('Red bars = potential vanishing gradients')

---
### Common Mistakes

| Mistake | Symptom | Fix |
|---------|---------|-----|
| Forgetting `optimizer.zero_grad()` | Gradients accumulate across steps, loss diverges | Call it at the start of every training loop iteration |
| Calling `backward()` twice | `RuntimeError: Trying to backward through the graph a second time` | Use `retain_graph=True` only if you truly need it |
| Detaching tensors before loss | Gradients do not flow, model doesn't learn | Never `.detach()` tensors that need gradient flow |
| Using Sigmoid in deep networks | Vanishing gradients (as shown above) | Use ReLU/GELU in hidden layers; Sigmoid only at output for binary classification |
| Very large learning rate | Loss explodes / NaN | Use gradient clipping: `nn.utils.clip_grad_norm_(model.parameters(), 1.0)` |

---
### Further Reading

- **3Blue1Brown** — *But what is a neural network?* and *Backpropagation calculus*: https://www.youtube.com/c/3blue1brown  
- **CS231n** — Backpropagation notes: https://cs231n.github.io/optimization-2/  
- **PyTorch autograd tutorial**: https://pytorch.org/tutorials/beginner/blitz/autograd_tutorial.html  
- **Karpathy's micrograd** — 25-line autograd engine for intuition: https://github.com/karpathy/micrograd

---
### Exercises

1. **Manual vs autograd**: Compute `dC/dw` by hand for `C = (sigmoid(w*2.0 + 0.5) - 1.0)^2`.
   Then verify your answer using PyTorch autograd.

2. **Vanishing gradients**: Swap the `nn.Sigmoid` activations in the gradient flow demo
   above for `nn.ReLU`. Re-run and compare the gradient norms. What do you observe?

3. **Gradient accumulation**: Run 4 forward-backward passes *without* calling
   `optimizer.zero_grad()` between them. Print `w.grad` after each pass.
   Why does the value keep growing? When might intentional gradient accumulation be useful?